In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd

df = pd.read_csv(
    "/mnt/c/Users/manav/OneDrive/Desktop/paitent health/Patient_Health_Records_Cleaned.csv"
)

print(df.head())

    Age  Gender         City   BMI  Heart_Rate  Cholesterol_Level  Diabetic  \
0  48.0       0       Boston  22.5        83.0              190.0         0   
1  48.0       0     New York  25.8        87.0              190.0         0   
2  48.0       0     New York  22.5        87.0              190.0         0   
3  80.0       0     New York  33.4        87.0              250.0         1   
4  48.0       1  Los Angeles  30.1        87.0              190.0         0   

   Smoker         Medications  Follow_Up Diagnosis_Code Has_Disease  \
0       1             Unknown       30.0            A00           0   
1       0             Unknown       14.0        Unknown     Unknown   
2       1  Aspirin; Metformin       30.0            B20     Unknown   
3       1             Unknown       30.0        Unknown     Unknown   
4       0             Unknown       14.0            I10           1   

   Systolic_BP  Diastolic_BP  
0        120.0          80.0  
1        120.0          80.0  
2    

In [3]:
# Create ML copy
df_ml = df.copy()

# Standardize Unknown values
df_ml['Has_Disease'] = df_ml['Has_Disease'].astype(str).str.strip()

df_ml['Has_Disease'] = df_ml['Has_Disease'].replace({
    'unknown': 'Unknown',
    'UNKNOWN': 'Unknown',
    'Unknown': 'Unknown'
})

# Remove Unknown target rows
df_ml = df_ml[df_ml['Has_Disease'] != 'Unknown'].copy()

# Check result
print(df_ml['Has_Disease'].value_counts())
print(df_ml.shape)

Has_Disease
0    2424
1    2386
Name: count, dtype: int64
(4810, 14)


In [4]:
df_ml.info()

<class 'pandas.DataFrame'>
Index: 4810 entries, 0 to 9492
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                4810 non-null   float64
 1   Gender             4810 non-null   int64  
 2   City               4810 non-null   str    
 3   BMI                4810 non-null   float64
 4   Heart_Rate         4810 non-null   float64
 5   Cholesterol_Level  4810 non-null   float64
 6   Diabetic           4810 non-null   int64  
 7   Smoker             4810 non-null   int64  
 8   Medications        4810 non-null   str    
 9   Follow_Up          4810 non-null   float64
 10  Diagnosis_Code     4810 non-null   str    
 11  Has_Disease        4810 non-null   str    
 12  Systolic_BP        4810 non-null   float64
 13  Diastolic_BP       4810 non-null   float64
dtypes: float64(7), int64(3), str(4)
memory usage: 563.7 KB


In [5]:
df_ml = df_ml.drop(['City', 'Follow_Up', 'Diagnosis_Code'], axis=1)

In [6]:
df_ml.info()

<class 'pandas.DataFrame'>
Index: 4810 entries, 0 to 9492
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Age                4810 non-null   float64
 1   Gender             4810 non-null   int64  
 2   BMI                4810 non-null   float64
 3   Heart_Rate         4810 non-null   float64
 4   Cholesterol_Level  4810 non-null   float64
 5   Diabetic           4810 non-null   int64  
 6   Smoker             4810 non-null   int64  
 7   Medications        4810 non-null   str    
 8   Has_Disease        4810 non-null   str    
 9   Systolic_BP        4810 non-null   float64
 10  Diastolic_BP       4810 non-null   float64
dtypes: float64(6), int64(3), str(2)
memory usage: 450.9 KB


In [8]:
# Identify numerical columns
numeric_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

# Identify categorical columns
categorical_cols = X_train.select_dtypes(
    include=['object', 'str', 'category', 'bool']
).columns

print("Numerical columns:")
print(numeric_cols.tolist())

print("\nCategorical columns:")
print(categorical_cols.tolist())

Numerical columns:
['Age', 'Gender', 'BMI', 'Heart_Rate', 'Cholesterol_Level', 'Diabetic', 'Smoker', 'Systolic_BP', 'Diastolic_BP']

Categorical columns:
['Medications']


In [9]:
print(X_train.dtypes)

Age                  float64
Gender                 int64
BMI                  float64
Heart_Rate           float64
Cholesterol_Level    float64
Diabetic               int64
Smoker                 int64
Medications              str
Systolic_BP          float64
Diastolic_BP         float64
dtype: object


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separate features and target
X = df_ml.drop(columns=['Has_Disease'])
y = df_ml['Has_Disease']

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Numerical features
numeric_cols = [
    'Age',
    'BMI',
    'Heart_Rate',
    'Cholesterol_Level',
    'Systolic_BP',
    'Diastolic_BP'
]

# Categorical/binary features
categorical_cols = [
    'Gender',
    'Diabetic',
    'Smoker',
    'Medications'
]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the same preprocessing
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (3848, 15)
Testing shape: (962, 15)


In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("Predictions:", y_pred[:10])

Predictions: ['1' '0' '0' '0' '1' '0' '0' '1' '0' '1']


In [12]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.4677754677754678

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.55      0.51       485
           1       0.46      0.38      0.42       477

    accuracy                           0.47       962
   macro avg       0.47      0.47      0.46       962
weighted avg       0.47      0.47      0.46       962



In [13]:
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Has_Disease
0    485
1    477
Name: count, dtype: int64
Has_Disease
0    0.504158
1    0.495842
Name: proportion, dtype: float64


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.48      0.55      0.51       485
           1       0.46      0.38      0.42       477

    accuracy                           0.47       962
   macro avg       0.47      0.47      0.46       962
weighted avg       0.47      0.47      0.46       962

Confusion Matrix:
[[267 218]
 [294 183]]


In [15]:
print(df_ml['Has_Disease'].value_counts())
print(df_ml['Has_Disease'].value_counts(normalize=True))

Has_Disease
0    2424
1    2386
Name: count, dtype: int64
Has_Disease
0    0.50395
1    0.49605
Name: proportion, dtype: float64


In [16]:
print(df_ml['Medications'].value_counts())

Medications
Unknown                  1917
Aspirin; Metformin       1903
Lisinopril; Metformin     990
Name: count, dtype: int64


In [17]:
print("Target:")
print(df_ml['Has_Disease'].value_counts())

print("\nTarget percentages:")
print(df_ml['Has_Disease'].value_counts(normalize=True))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Target:
Has_Disease
0    2424
1    2386
Name: count, dtype: int64

Target percentages:
Has_Disease
0    0.50395
1    0.49605
Name: proportion, dtype: float64

Classification report:
              precision    recall  f1-score   support

           0       0.48      0.55      0.51       485
           1       0.46      0.38      0.42       477

    accuracy                           0.47       962
   macro avg       0.47      0.47      0.46       962
weighted avg       0.47      0.47      0.46       962


Confusion matrix:
[[267 218]
 [294 183]]


In [18]:
print(df_ml.groupby('Has_Disease').mean(numeric_only=True).T)

Has_Disease                 0           1
Age                 50.491749   50.132020
Gender               0.333746    0.322297
BMI                 23.043234   23.114334
Heart_Rate          86.272277   86.686085
Cholesterol_Level  202.054455  201.894384
Diabetic             0.322195    0.330260
Smoker               0.510314    0.508382
Systolic_BP        120.931106  121.082146
Diastolic_BP        79.949670   80.099329


In [20]:
# Make a temporary numeric target for correlation
df_corr = df_ml.copy()

df_corr['Has_Disease'] = pd.to_numeric(
    df_corr['Has_Disease'],
    errors='coerce'
)

# Select numerical columns
corr = df_corr.select_dtypes(include='number').corr()

# Show correlation with target
print(
    corr['Has_Disease']
    .sort_values(ascending=False)
)

Has_Disease          1.000000
Heart_Rate           0.019496
Diastolic_BP         0.013609
BMI                  0.011580
Diabetic             0.008601
Systolic_BP          0.007909
Smoker              -0.001932
Cholesterol_Level   -0.003337
Gender              -0.012192
Age                 -0.015022
Name: Has_Disease, dtype: float64


In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

rf_pred = rf_model.predict(X_test_processed)

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.5031185031185031

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.48      0.49       485
           1       0.50      0.53      0.51       477

    accuracy                           0.50       962
   macro avg       0.50      0.50      0.50       962
weighted avg       0.50      0.50      0.50       962



In [22]:
# Compare numerical features by disease status

comparison = df_ml.groupby('Has_Disease')[
    ['Age', 'BMI', 'Heart_Rate',
     'Cholesterol_Level', 'Systolic_BP',
     'Diastolic_BP']
].mean()

print(comparison)

                   Age        BMI  Heart_Rate  Cholesterol_Level  Systolic_BP  \
Has_Disease                                                                     
0            50.491749  23.043234   86.272277         202.054455   120.931106   
1            50.132020  23.114334   86.686085         201.894384   121.082146   

             Diastolic_BP  
Has_Disease                
0               79.949670  
1               80.099329  


In [23]:
for col in ['Gender', 'Diabetic', 'Smoker', 'Medications']:
    print(f"\n===== {col} =====")
    print(
        pd.crosstab(
            df_ml[col],
            df_ml['Has_Disease'],
            normalize='index'
        ).round(3)
    )


===== Gender =====
Has_Disease      0      1
Gender                   
0            0.500  0.500
1            0.513  0.487

===== Diabetic =====
Has_Disease      0      1
Diabetic                 
0            0.507  0.493
1            0.498  0.502

===== Smoker =====
Has_Disease      0      1
Smoker                   
0            0.503  0.497
1            0.505  0.495

===== Medications =====
Has_Disease                0      1
Medications                        
Aspirin; Metformin     0.509  0.491
Lisinopril; Metformin  0.492  0.508
Unknown                0.505  0.495


In [24]:
# Check disease rate for every feature
for col in [
    'Age',
    'BMI',
    'Heart_Rate',
    'Cholesterol_Level',
    'Systolic_BP',
    'Diastolic_BP'
]:
    print(f"\n===== {col} =====")
    
    print(
        df_ml.groupby('Has_Disease')[col]
        .agg(['mean', 'std', 'min', 'max'])
        .round(2)
    )


===== Age =====
              mean    std   min    max
Has_Disease                           
0            50.49  12.45  18.0  100.0
1            50.13  11.47  18.0  100.0

===== BMI =====
              mean   std   min   max
Has_Disease                         
0            23.04  3.13  15.0  35.0
1            23.11  3.01  15.3  35.0

===== Heart_Rate =====
              mean    std   min    max
Has_Disease                           
0            86.27  10.49  50.0  120.0
1            86.69  10.73  50.0  120.0

===== Cholesterol_Level =====
               mean    std    min    max
Has_Disease                             
0            202.05  24.05  190.0  250.0
1            201.89  23.93  190.0  250.0

===== Systolic_BP =====
               mean   std   min    max
Has_Disease                           
0            120.93  9.89  90.0  160.0
1            121.08  9.20  90.0  160.0

===== Diastolic_BP =====
              mean   std   min    max
Has_Disease                          
0   

In [25]:
from scipy.stats import ttest_ind

numeric_cols = [
    'Age',
    'BMI',
    'Heart_Rate',
    'Cholesterol_Level',
    'Systolic_BP',
    'Diastolic_BP'
]

for col in numeric_cols:
    group0 = df_ml[df_ml['Has_Disease'] == 0][col]
    group1 = df_ml[df_ml['Has_Disease'] == 1][col]

    statistic, p_value = ttest_ind(group0, group1)

    print(f"{col:20} p-value = {p_value:.4f}")

Age                  p-value = nan
BMI                  p-value = nan
Heart_Rate           p-value = nan
Cholesterol_Level    p-value = nan
Systolic_BP          p-value = nan
Diastolic_BP         p-value = nan


/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample argu

In [26]:
from scipy.stats import chi2_contingency

categorical_cols = [
    'Gender',
    'Diabetic',
    'Smoker',
    'Medications'
]

for col in categorical_cols:

    table = pd.crosstab(
        df_ml[col],
        df_ml['Has_Disease']
    )

    chi2, p_value, dof, expected = chi2_contingency(table)

    print(f"{col:20} p-value = {p_value:.4f}")

Gender               p-value = 0.4151
Diabetic             p-value = 0.5715
Smoker               p-value = 0.9163
Medications          p-value = 0.6836


In [27]:
from scipy.stats import ttest_ind

numeric_cols = [
    'Age',
    'BMI',
    'Heart_Rate',
    'Cholesterol_Level',
    'Systolic_BP',
    'Diastolic_BP'
]

for col in numeric_cols:
    group0 = df_ml[df_ml['Has_Disease'] == 0][col]
    group1 = df_ml[df_ml['Has_Disease'] == 1][col]

    statistic, p_value = ttest_ind(group0, group1)

    print(f"{col:20} p-value = {p_value:.4f}")

Age                  p-value = nan
BMI                  p-value = nan
Heart_Rate           p-value = nan
Cholesterol_Level    p-value = nan
Systolic_BP          p-value = nan
Diastolic_BP         p-value = nan


/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  statistic, p_value = ttest_ind(group0, group1)
/tmp/ipykernel_5214/774071264.py:16: SmallSampleWarning: One or more sample argu